# Ground Truth : paliers 1 et 2 sur un GPU T4

Ce notebook produit tous les résultats des paliers 1 et 2 à partir du code de ton dépôt.
**Avant de commencer :** Exécution → Modifier le type d'exécution → **GPU T4**. Lance ensuite les cellules dans l'ordre (Maj + Entrée).
Durée totale : environ 2 h, surtout de l'attente. Ne ferme pas l'onglet.

| Étape | Produit |
|---|---|
| 0. Installation | dépôt, dépendances, modèles, corpus |
| 1. Palier 1 | BM25 maison, BM25 vs dense, RAG de base, 5 échecs |
| 2. Palier 2 | index complet + variantes, ablation de la recherche, calibration du seuil, comparaison des parseurs |
| 3. Soumission v1 | `submissions/v1/…json` + fichiers à rapatrier |

## 0. Installation

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
REPO = "https://github.com/YanisYounsi-code/Ground-Truth-un-RAG-qui-ne-ment-pas.git"
!rm -rf /content/gt && git clone -q $REPO /content/gt
%cd /content/gt
!git log --oneline -3

In [ ]:
# Dépendances, Tesseract (OCR) et poids des modèles : environ 10 minutes
!bash setup.sh
!python corpus/download_corpus.py
!ls corpus/pdf | wc -l

## 1. Palier 1 : RAG from scratch

**1.1 BM25 maison sur les pages** (une page = un document), vérifié contre bm25s.

In [ ]:
!python -m scratch.extract
!python -m scratch.check_bm25s
!python -m scratch.eval_bm25

**1.2 Index du palier 1** : texte brut PyMuPDF, chunker naïf (250 mots, recouvrement 50), pas d'OCR, pas de tableaux.
Puis **BM25 seul vs dense seul** sur ces chunks.

In [ ]:
!GT_OVERRIDE='{"parser": "pymupdf", "ocr": false, "chunking": "naive"}' python -m pipeline index --pdf-dir corpus/pdf --index-dir /content/idx/palier1
!python -m pipeline.ablation --out des_tests/palier1_recherche.md \
    --config bm25_seul=/content/idx/palier1:bm25:norerank \
    --config dense_seul=/content/idx/palier1:dense:norerank \
    --config bm25+dense_rrf=/content/idx/palier1:hybrid:norerank

**1.3 RAG de base** : BM25 sur les chunks naïfs, 6 passages, le générateur officiel, une citation. Sans reranker ni porte d'abstention.

In [ ]:
!python -m pipeline answer --index-dir /content/idx/palier1 --questions data/validation_v1.json \
    --out runs/val1_rag_base.json --mode bm25 --no-rerank --no-gate
!python -m groundtruth.evaluate --pred runs/val1_rag_base.json --gold data/validation_v1.json --judge hf --out runs/val1_rag_base_report.json

**1.4 Les 5 échecs à commenter** (écrits dans `des_tests/echecs.md`, la cause est déjà identifiée : recherche ou génération).

In [ ]:
!python -m pipeline.failures --pred runs/val1_rag_base.json --report runs/val1_rag_base_report.json --out des_tests/echecs.md

## 2. Palier 2 : le pipeline

**2.1 Index complet** : PyMuPDF + tableaux en Markdown, OCR Tesseract des scans, chunking contextuel, embeddings bge-m3.
Le temps affiché compte pour la limite de 2 h.

In [ ]:
!time python -m pipeline index --pdf-dir corpus/pdf --index-dir /content/idx/complet

**2.2 Variantes pour l'ablation** (elles réutilisent les pages déjà lues, sauf la variante sans tableaux).

In [ ]:
!GT_OVERRIDE='{"chunking": "structure"}' python -m pipeline index --pdf-dir corpus/pdf --index-dir /content/idx/structure --pages-from /content/idx/complet
!GT_OVERRIDE='{"chunking": "naive"}' python -m pipeline index --pdf-dir corpus/pdf --index-dir /content/idx/naif --pages-from /content/idx/complet
!GT_OVERRIDE='{"ocr": false}' python -m pipeline index --pdf-dir corpus/pdf --index-dir /content/idx/sans_ocr --pages-from /content/idx/complet
!GT_OVERRIDE='{"parser": "pymupdf"}' python -m pipeline index --pdf-dir corpus/pdf --index-dir /content/idx/sans_tableaux

**2.3 Ablation de la recherche** : le pipeline complet, puis en retirant une brique à la fois.

In [ ]:
!python -m pipeline.ablation --out des_tests/ablation_recherche.md \
    --config complet=/content/idx/complet:hybrid:rerank \
    --config sans_reranker=/content/idx/complet:hybrid:norerank \
    --config sans_dense=/content/idx/complet:bm25:rerank \
    --config sans_bm25=/content/idx/complet:dense:rerank \
    --config chunking_structure=/content/idx/structure:hybrid:rerank \
    --config chunking_naif=/content/idx/naif:hybrid:rerank \
    --config sans_ocr=/content/idx/sans_ocr:hybrid:rerank \
    --config sans_tableaux=/content/idx/sans_tableaux:hybrid:rerank

**2.4 Pipeline complet sur la validation, sans porte**, puis **calibration du seuil d'abstention** :
pour chaque seuil possible, le score de confiance qu'on aurait obtenu. Le meilleur seuil est écrit dans `pipeline/settings.json`.

In [ ]:
!python -m pipeline answer --index-dir /content/idx/complet --questions data/validation_v1.json --out runs/val1_complet_sansporte.json --no-gate
!python -m groundtruth.evaluate --pred runs/val1_complet_sansporte.json --gold data/validation_v1.json --judge hf --out runs/val1_complet_sansporte_report.json
!python -m pipeline.calibrate --pred runs/val1_complet_sansporte.json --report runs/val1_complet_sansporte_report.json --csv des_tests/calibration.csv
!cat pipeline/settings.json

**2.5 Vérification avec la porte** (réutilise la recherche déjà faite : seules les questions au-dessus du seuil appellent le LLM).

In [ ]:
!python -m pipeline generate --questions data/validation_v1.json --out runs/val1_complet.json \
    --retrieval runs/val1_complet_sansporte.retrieval.json
!python -m groundtruth.evaluate --pred runs/val1_complet.json --gold data/validation_v1.json --judge hf --out runs/val1_complet_report.json

In [ ]:
# Courbe de calibration (pour le rapport d'ablation)
import pandas as pd, matplotlib.pyplot as plt, json
c = pd.read_csv("des_tests/calibration.csv", sep=";")
s = json.load(open("pipeline/settings.json"))["threshold"]
plt.figure(figsize=(7, 4)); plt.plot(c.seuil, c.score_confiance)
plt.axvline(s, ls="--", c="gray"); plt.xlabel("seuil sur le score du reranker"); plt.ylabel("score de confiance (validation v1)")
plt.title(f"Seuil retenu : {s}"); plt.grid(alpha=.3); plt.savefig("des_tests/calibration.png", dpi=150, bbox_inches="tight"); plt.show()

**2.6 Comparaison des parseurs sur 30 pages difficiles.** L'annotation se fait à la main, sur ton ordi :
`des_tests/parsing_annotations.json`, avec les images dans `runs/parsing_png/`. Une fois le fichier annoté et poussé sur GitHub,
relance la cellule 0 (pour récupérer la dernière version), puis celle-ci.

In [ ]:
!git pull -q
!python -m pipeline.parsing_bench eval

## 3. Soumission v1 et rapatriement des fichiers

Le pipeline complet, avec le seuil calibré, sur les 200 questions de classement. Puis préparation de la soumission.

In [ ]:
!python -m pipeline answer --index-dir /content/idx/complet --questions data/classement_v1_questions.json --out runs/classement_v1.json
!python -m groundtruth.submit --pred runs/classement_v1.json --phase v1

In [ ]:
# Télécharge tout ce qu'il faut remettre dans ton dépôt (une archive zip)
!zip -q -r /content/a_rapatrier.zip submissions des_tests pipeline/settings.json runs/*report.json runs/generator_log.jsonl
from google.colab import files
files.download("/content/a_rapatrier.zip")

Dézippe `a_rapatrier.zip` dans ton dossier `ground-truth-kit` (en remplaçant les fichiers), puis :
```
git add submissions des_tests pipeline/settings.json
git commit -m "paliers 1 et 2 : résultats, seuil calibré, soumission v1"
git push
```